# Extractor multicanal YOLO — primera tanda

Versión preparada para compartir y con metadatos autocontenidos.

## Uso

1. Ejecutar la celda **Configuración** y cambiar únicamente `BASE_DIR`, `LABEL_DIR` y `OUTPUT_DIR` si las carpetas están en otra ubicación.
2. Ejecutar la celda **Extractor**.
3. Ejecutar **Procesar recortes**.

El archivo `metadata_recortes.json` generado en cada carpeta de salida es **autocontenido**: incorpora las cajas YOLO utilizadas y el contenido completo del JSON fuente de cada canal. Las rutas de la computadora donde se hizo la extracción no son necesarias para interpretar los datos.

La convención de recorte se mantiene: **X de la imagen → eje 0/pings de la matriz** y **Y de la imagen → eje 1/muestras/profundidad**.


In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURACIÓN — PARA USARLO EN OTRA COMPUTADORA,
# MODIFICAR SOLAMENTE ESTAS TRES RUTAS
# ============================================================

BASE_DIR = Path(r"/mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/todas_imagenes_yolo_v4_fish_v6_MAS")
LABEL_DIR = BASE_DIR / "etiquetas_p_cluster_mas_datos" / "obj_train_data"      #donde estan las coordenadas de los recortes yolo bajadas de cvat
OUTPUT_DIR = Path(r"/mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/recortes_desde_yolo_segunda_tanda_v2")

# Si True, borra OUTPUT_DIR antes de regenerar todos los recortes.
BORRAR_SALIDA_ANTERIOR = True

# metadata_recortes.json ya es autocontenido.
# Esto solo decide si además se guarda una copia separada de cada JSON fuente.
COPIAR_JSON_ORIGINALES = True

VERSION_METADATA = "SEAMAN_RECORTES_AUTOCONTENIDO_v2"

print("BASE_DIR :", BASE_DIR)
print("LABEL_DIR:", LABEL_DIR)
print("OUTPUT   :", OUTPUT_DIR)


BASE_DIR : /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/todas_imagenes_yolo_v4_fish_v6_MAS
LABEL_DIR: /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/todas_imagenes_yolo_v4_fish_v6_MAS/etiquetas_p_cluster_mas_datos/obj_train_data
OUTPUT   : /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/recortes_desde_yolo_segunda_tanda_v2


In [11]:
# -*- coding: utf-8 -*-
"""
Extractor de recortes YOLO multicanal corregido.

Corrección principal:
- Las matrices completas guardadas desde FishID tienen forma usual (pings, muestras):
      eje 0 = columna temporal / ping  -> corresponde al eje X del JPG
      eje 1 = muestra / profundidad    -> corresponde al eje Y del JPG
- Por lo tanto, una caja YOLO marcada sobre el JPG debe convertirse así:
      x JPG -> eje 0 de la matriz
      y JPG -> eje 1 de la matriz
  y NO como mat[y, x].

Esto evita los recortes (181,15) con rayas; ahora deberían salir como aprox.
(ancho_en_pings, alto_en_muestras), por ejemplo (13,216), coherente con la caja marcada.

Además:
- Detecta si la etiqueta fue marcada en CH1, CH2, CH3, CH4 o VIRTUAL.
- Si fue marcada en VIRTUAL, extrae CH1-CH4 y también VIRTUAL si existe.
- En SplitBeam no inventa S3 virtual: el virtual suele tener S5/S7 solamente.
"""

import json
import re
import shutil
from pathlib import Path

from PIL import Image
import numpy as np

# ============================================================
# ============================================================
# CONFIGURACIÓN GENERAL DEL EXTRACTOR (no editar normalmente)
# ============================================================

# Esta es la orientación correcta para las matrices generadas por tu lector:
# matriz.shape = (pings, muestras)
MATRIZ_EJE0_ES_X_PING = True

CANALES_REALES = ["CH1", "CH2", "CH3", "CH4"]
CANALES_TODOS = CANALES_REALES + ["VIRTUAL"]

MATRICES_SUFFIXES = [
    "Data1_S2_FILTERED.npy",
    "Data2_S3_CORRELATOR_DETECTOR.npy",
    "Data3_S5_DB_WITHOUT_TVG.npy",
    "Data4_S7_DB.npy",
    "Data4_S7_DB_VIRTUAL.npy",
    "Data3_S5_DB_WITHOUT_TVG_VIRTUAL.npy",
]

EXT_IMAGENES = [".jpg", ".jpeg", ".png", ".bmp"]


# ============================================================
# UTILIDADES
# ============================================================

def corregir_stem_duplicado(stem: str) -> str:
    """Corrige prefijos duplicados del tipo 17_06_17_06_Seaman..."""
    partes = stem.split("_Seaman_Fish_ID_", 1)
    if len(partes) != 2:
        return stem
    prefijos, resto = partes
    tokens = prefijos.split("_")
    n = len(tokens)
    for mitad in range(1, n // 2 + 1):
        if tokens[:mitad] == tokens[mitad:2 * mitad]:
            prefijo_simple = "_".join(tokens[:mitad])
            return f"{prefijo_simple}_Seaman_Fish_ID_{resto}"
    return stem


def detectar_canal_en_stem(stem: str):
    """Devuelve (base_sin_canal, canal_detectado)."""
    st = corregir_stem_duplicado(stem)
    m = re.search(r"_(CH[1-4])$", st, flags=re.IGNORECASE)
    if m:
        canal = m.group(1).upper()
        return st[: -len(canal) - 1], canal

    for suf in ["_VIRTUAL_S7", "_VIRTUAL", "_SPB", "_SPLITBEAM", "_SPLIT_BEAM"]:
        if st.upper().endswith(suf):
            return st[: -len(suf)], "VIRTUAL"

    return st, "DESCONOCIDO"


def candidatos_stem_canal(base: str, canal: str):
    base = corregir_stem_duplicado(base)
    canal = canal.upper()
    if canal in CANALES_REALES:
        return [f"{base}_{canal}"]
    if canal == "VIRTUAL":
        return [
            f"{base}_VIRTUAL",
            f"{base}_VIRTUAL_S7",
            f"{base}_SPB",
            f"{base}_SPLITBEAM",
            f"{base}_SPLIT_BEAM",
        ]
    return [base]


def buscar_imagen_por_stems(stems):
    for stem in stems:
        for st in [stem, corregir_stem_duplicado(stem)]:
            for ext in EXT_IMAGENES:
                p = BASE_DIR / f"{st}{ext}"
                if p.exists():
                    return p, st
    return None, stems[0] if stems else None


def buscar_imagen_canal(base: str, canal: str):
    return buscar_imagen_por_stems(candidatos_stem_canal(base, canal))


def buscar_json(stem: str):
    p = BASE_DIR / f"{stem}.json"
    return p if p.exists() else None


def cargar_matrices(stem: str):
    matrices = {}
    if stem is None:
        return matrices
    for suffix in MATRICES_SUFFIXES:
        mat_path = BASE_DIR / f"{stem}_{suffix}"
        if mat_path.exists():
            nombre = suffix.replace(".npy", "")
            matrices[nombre] = {
                "path": mat_path,
                "array": np.load(mat_path, allow_pickle=False),
            }
    return matrices


def canal_tiene_datos(base: str, canal: str):
    image_path, stem_real = buscar_imagen_canal(base, canal)
    matrices = cargar_matrices(stem_real) if stem_real else {}
    return image_path is not None or len(matrices) > 0


def detectar_canales_disponibles(base: str, canal_marcado: str):
    canales = []
    for canal in CANALES_REALES:
        if canal_tiene_datos(base, canal):
            canales.append(canal)
    if canal_tiene_datos(base, "VIRTUAL"):
        canales.append("VIRTUAL")
    if not canales and canal_marcado in CANALES_TODOS:
        canales = [canal_marcado]
    return canales


# ============================================================
# COORDENADAS YOLO Y RECORTE DE MATRICES
# ============================================================

def yolo_to_pixels(xc, yc, w, h, img_w, img_h):
    cx = xc * img_w
    cy = yc * img_h
    bw = w * img_w
    bh = h * img_h

    x1 = int(round(cx - bw / 2))
    y1 = int(round(cy - bh / 2))
    x2 = int(round(cx + bw / 2))
    y2 = int(round(cy + bh / 2))

    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w))
    y2 = max(0, min(y2, img_h))
    return x1, y1, x2, y2


def recortar_matriz_corregida(mat, x1, y1, x2, y2, img_w, img_h):
    """
    Recorta una matriz usando la geometría correcta.

    Caso FishID usado acá:
      mat.shape = (pings, muestras)
      x del JPG -> pings  -> eje 0
      y del JPG -> muestras -> eje 1

    Devuelve crop y metadatos.
    """
    if mat.ndim < 2:
        return None, {"error": "matriz con menos de 2 dimensiones"}

    n0, n1 = mat.shape[:2]

    if MATRIZ_EJE0_ES_X_PING:
        # eje 0: X/ping; eje 1: Y/muestras
        ix1 = int(round(x1 * n0 / img_w))
        ix2 = int(round(x2 * n0 / img_w))
        iy1 = int(round(y1 * n1 / img_h))
        iy2 = int(round(y2 * n1 / img_h))

        ix1 = max(0, min(ix1, n0 - 1))
        ix2 = max(0, min(ix2, n0))
        iy1 = max(0, min(iy1, n1 - 1))
        iy2 = max(0, min(iy2, n1))

        if ix2 <= ix1 or iy2 <= iy1:
            return None, {
                "error": "recorte vacío",
                "orientacion": "eje0_x_ping_eje1_y_muestra",
                "bbox_matriz": {"i_ping1": ix1, "i_ping2": ix2, "i_muestra1": iy1, "i_muestra2": iy2},
            }

        crop = mat[ix1:ix2, iy1:iy2, ...]
        return crop, {
            "orientacion": "eje0_x_ping_eje1_y_muestra",
            "bbox_matriz": {"i_ping1": ix1, "i_ping2": ix2, "i_muestra1": iy1, "i_muestra2": iy2},
            "shape_matriz_original": list(mat.shape),
            "shape_recorte": list(crop.shape),
        }

    # Modo alternativo por si alguna matriz viniera como (muestras, pings)
    mh, mw = mat.shape[:2]
    mx1 = int(round(x1 * mw / img_w))
    mx2 = int(round(x2 * mw / img_w))
    my1 = int(round(y1 * mh / img_h))
    my2 = int(round(y2 * mh / img_h))

    mx1 = max(0, min(mx1, mw - 1))
    mx2 = max(0, min(mx2, mw))
    my1 = max(0, min(my1, mh - 1))
    my2 = max(0, min(my2, mh))

    if mx2 <= mx1 or my2 <= my1:
        return None, {"error": "recorte vacío", "orientacion": "yx_escalado"}
    crop = mat[my1:my2, mx1:mx2, ...]
    return crop, {
        "orientacion": "yx_escalado",
        "bbox_matriz": {"fila1": my1, "fila2": my2, "col1": mx1, "col2": mx2},
        "shape_matriz_original": list(mat.shape),
        "shape_recorte": list(crop.shape),
    }



# ============================================================
# GEOMETRÍA VERTICAL Y PROFUNDIDAD DEL RECORTE
# ============================================================

def cargar_json_seguro(path_json):
    if path_json is None or not Path(path_json).exists():
        return None

    try:
        with open(path_json, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[AVISO] No pude leer JSON {path_json}: {e}")
        return None


def extraer_geometria_imagen(json_original):
    """
    Extrae la nueva sección:

        processed_view.image_geometry

    agregada por el programa que genera las imágenes completas.
    """
    if not isinstance(json_original, dict):
        return None

    processed_view = json_original.get("processed_view", {})
    if not isinstance(processed_view, dict):
        return None

    geometria = processed_view.get("image_geometry")
    return geometria if isinstance(geometria, dict) else None


def valor_numerico_valido(valor):
    """
    En archivos SEAMAN, cualquier valor -1 significa dato no disponible.
    """
    if valor is None:
        return None

    try:
        valor = float(valor)
    except Exception:
        return None

    if not np.isfinite(valor) or valor == -1.0:
        return None

    return valor


def profundidad_desde_pixel_y(y_px, geometria):
    """
    Convierte una coordenada vertical del JPG completo a profundidad.

    Casos soportados:

    1) bottom_aligned:
       height_m = top_height_above_bottom_m
                  - y_px * meters_per_vertical_pixel

       depth_m = bottom_depth_m_used - height_m

    2) surface_to_bottom_cut:
       depth_m = y_px * meters_per_vertical_pixel
    """
    if not isinstance(geometria, dict):
        return None

    referencia = geometria.get("vertical_reference")
    metros_por_pixel = valor_numerico_valido(
        geometria.get("meters_per_vertical_pixel")
    )

    if metros_por_pixel is None:
        return None

    y_px = float(y_px)

    if referencia == "bottom_aligned":
        altura_superior = valor_numerico_valido(
            geometria.get("top_height_above_bottom_m")
        )
        profundidad_fondo = valor_numerico_valido(
            geometria.get("bottom_depth_m_used")
        )

        if altura_superior is None or profundidad_fondo is None:
            return None

        altura_sobre_fondo = (
            altura_superior - y_px * metros_por_pixel
        )

        return float(
            profundidad_fondo - altura_sobre_fondo
        )

    if referencia == "surface_to_bottom_cut":
        profundidad_superior = valor_numerico_valido(
            geometria.get("top_depth_m_approx")
        )

        if profundidad_superior is None:
            profundidad_superior = 0.0

        return float(
            profundidad_superior + y_px * metros_por_pixel
        )

    return None


def altura_sobre_fondo_desde_pixel_y(y_px, geometria):
    """
    Devuelve altura sobre el fondo cuando la geometría permite calcularla.
    """
    if not isinstance(geometria, dict):
        return None

    referencia = geometria.get("vertical_reference")
    metros_por_pixel = valor_numerico_valido(
        geometria.get("meters_per_vertical_pixel")
    )

    if metros_por_pixel is None:
        return None

    if referencia == "bottom_aligned":
        altura_superior = valor_numerico_valido(
            geometria.get("top_height_above_bottom_m")
        )

        if altura_superior is None:
            return None

        return float(
            altura_superior - float(y_px) * metros_por_pixel
        )

    profundidad_fondo = valor_numerico_valido(
        geometria.get("bottom_depth_m_approx")
    )
    profundidad = profundidad_desde_pixel_y(
        y_px,
        geometria,
    )

    if profundidad_fondo is None or profundidad is None:
        return None

    return float(profundidad_fondo - profundidad)


def calcular_profundidad_recorte(
    y1_px,
    y2_px,
    geometria,
):
    """
    Calcula profundidad superior, central e inferior del recorte,
    su altura vertical en metros y su distancia al fondo.

    Las coordenadas y1/y2 deben corresponder al JPG del canal,
    no necesariamente al JPG usado como referencia YOLO.
    """
    if not isinstance(geometria, dict):
        return {
            "disponible": False,
            "motivo": (
                "El JSON original no contiene "
                "processed_view.image_geometry"
            ),
        }

    y_centro = (float(y1_px) + float(y2_px)) / 2.0

    profundidad_superior = profundidad_desde_pixel_y(
        y1_px,
        geometria,
    )
    profundidad_centro = profundidad_desde_pixel_y(
        y_centro,
        geometria,
    )
    profundidad_inferior = profundidad_desde_pixel_y(
        y2_px,
        geometria,
    )

    metros_por_pixel = valor_numerico_valido(
        geometria.get("meters_per_vertical_pixel")
    )

    altura_vertical = None
    if metros_por_pixel is not None:
        altura_vertical = abs(
            float(y2_px) - float(y1_px)
        ) * metros_por_pixel

    altura_centro_sobre_fondo = (
        altura_sobre_fondo_desde_pixel_y(
            y_centro,
            geometria,
        )
    )

    disponible = all(
        valor is not None
        for valor in [
            profundidad_superior,
            profundidad_centro,
            profundidad_inferior,
        ]
    )

    return {
        "disponible": bool(disponible),
        "referencia_vertical":
            geometria.get("vertical_reference"),
        "y_superior_px": int(y1_px),
        "y_centro_px": float(y_centro),
        "y_inferior_px": int(y2_px),
        "profundidad_superior_m": profundidad_superior,
        "profundidad_centro_m": profundidad_centro,
        "profundidad_inferior_m": profundidad_inferior,
        "altura_vertical_recorte_m": altura_vertical,
        "altura_centro_sobre_fondo_m":
            altura_centro_sobre_fondo,
        "metros_por_pixel_vertical": metros_por_pixel,
    }


# ============================================================
# PROCESAMIENTO
# ============================================================

def procesar_canal(base, canal, lines, out_folder, canal_marcado, source_img_size):
    image_path, stem_real = buscar_imagen_canal(base, canal)
    matrices = cargar_matrices(stem_real) if stem_real else {}
    json_path = buscar_json(stem_real) if stem_real else None
    json_original = cargar_json_seguro(json_path)
    geometria_imagen = extraer_geometria_imagen(
        json_original
    )

    if image_path is None and not matrices:
        print(f"[AVISO] No encontré imagen ni matrices para {canal}: base={base}")
        return None

    img = None
    if image_path is not None:
        img = np.array(Image.open(image_path))
        img_h, img_w = img.shape[:2]
    else:
        img_w, img_h = source_img_size

    # Para convertir coordenadas de matriz usamos el tamaño de la imagen donde se marcó YOLO.
    ref_img_w, ref_img_h = source_img_size

    metadata_canal = {
        "canal": canal,
        "canal_marcado_en_yolo": canal_marcado,
        "stem_usado": stem_real,

        # Estos nombres sirven para trazabilidad, pero NO son necesarios para
        # interpretar metadata_recortes.json.
        "archivo_imagen_fuente": image_path.name if image_path else None,
        "archivo_json_fuente": json_path.name if json_path else None,

        # AUTOCONTENIDO: se incorpora completo el JSON que acompañaba a la
        # imagen/canal original. Así un tercero no necesita recuperar ese
        # archivo para conocer los metadatos de adquisición/procesamiento.
        "metadata_fuente": json_original,
        "image_geometry_original": geometria_imagen,

        "imagen_canal_w_h": [img_w, img_h],
        "imagen_referencia_yolo_w_h": [ref_img_w, ref_img_h],
        "matrices_encontradas": list(matrices.keys()),
        "recortes": [],
    }

    # Se conserva, opcionalmente, una copia separada del JSON original por
    # comodidad humana. El metadata_recortes.json NO depende de esta copia.
    if COPIAR_JSON_ORIGINALES and json_path is not None:
        destino_json = out_folder / f"{canal}_{json_path.name}"
        if not destino_json.exists():
            shutil.copy2(json_path, destino_json)

    for idx, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) < 5:
            continue

        class_id = int(float(parts[0]))
        xc, yc, w, h = map(float, parts[1:5])

        # Coordenadas sobre la imagen de referencia marcada
        x1_ref, y1_ref, x2_ref, y2_ref = yolo_to_pixels(xc, yc, w, h, ref_img_w, ref_img_h)

        # Coordenadas equivalentes para recortar el JPG de este canal
        x1_img, y1_img, x2_img, y2_img = yolo_to_pixels(xc, yc, w, h, img_w, img_h)

        if x2_ref <= x1_ref or y2_ref <= y1_ref:
            continue

        recorte_id = f"recorte_{idx:04d}"

        img_crop_name = None
        if img is not None:
            img_crop = img[y1_img:y2_img, x1_img:x2_img]
            img_crop_name = f"{recorte_id}_marcado_{canal_marcado}_extraido_{canal}.jpg"
            Image.fromarray(img_crop).save(out_folder / img_crop_name)

        matriz_outputs = {}
        for mat_name, mat_info in matrices.items():
            crop, info = recortar_matriz_corregida(
                mat_info["array"],
                x1_ref, y1_ref, x2_ref, y2_ref,
                ref_img_w, ref_img_h,
            )

            if crop is None:
                matriz_outputs[mat_name] = {"guardado": False, **info}
                continue

            out_name = f"{recorte_id}_marcado_{canal_marcado}_extraido_{canal}_{mat_name}.npy"
            np.save(out_folder / out_name, crop)

            matriz_outputs[mat_name] = {
                "guardado": True,
                "archivo": out_name,
                **info,
            }

        profundidad_recorte = calcular_profundidad_recorte(
            y1_px=y1_img,
            y2_px=y2_img,
            geometria=geometria_imagen,
        )

        metadata_canal["recortes"].append({
            "id": recorte_id,
            "archivo_imagen": img_crop_name,
            "class_id": class_id,
            "bbox_yolo_normalizado": {
                "centro_x": xc,
                "centro_y": yc,
                "ancho": w,
                "alto": h,
            },
            "bbox_pixeles_en_imagen_referencia_yolo": {
                "x1": x1_ref, "y1": y1_ref, "x2": x2_ref, "y2": y2_ref,
                "ancho": x2_ref - x1_ref, "alto": y2_ref - y1_ref,
            },
            "bbox_pixeles_en_jpg_canal": {
                "x1": x1_img, "y1": y1_img, "x2": x2_img, "y2": y2_img,
                "ancho": x2_img - x1_img, "alto": y2_img - y1_img,
            },
            "profundidad_recorte": profundidad_recorte,
            "matrices": matriz_outputs,
        })

    return metadata_canal


def serializar_etiquetas_yolo(lines):
    """Convierte las líneas YOLO en una estructura JSON explícita y estable."""
    etiquetas = []
    for idx, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) < 5:
            continue
        class_id = int(float(parts[0]))
        xc, yc, w, h = map(float, parts[1:5])
        etiquetas.append({
            "id": f"recorte_{idx:04d}",
            "class_id": class_id,
            "centro_x": xc,
            "centro_y": yc,
            "ancho": w,
            "alto": h,
            "linea_yolo_original": line,
        })
    return etiquetas


# ============================================================
# MAIN
# ============================================================

def main():
    # Un usuario externo solo necesita configurar BASE_DIR, LABEL_DIR y
    # OUTPUT_DIR en la celda de configuración.
    if not BASE_DIR.is_dir():
        raise FileNotFoundError(f"No existe BASE_DIR: {BASE_DIR}")
    if not LABEL_DIR.is_dir():
        raise FileNotFoundError(f"No existe LABEL_DIR: {LABEL_DIR}")

    print("BASE_DIR :", BASE_DIR)
    print("LABEL_DIR:", LABEL_DIR)
    print("OUTPUT   :", OUTPUT_DIR)
    print()

    if BORRAR_SALIDA_ANTERIOR and OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    label_files = sorted(LABEL_DIR.glob("*.txt"))

    print(f"Etiquetas YOLO encontradas: {len(label_files)}")
    print(f"Buscando etiquetas en: {LABEL_DIR}")
    print(f"Salida: {OUTPUT_DIR}")
    print("Transformación matriz: x JPG -> eje 0/pings, y JPG -> eje 1/muestras")

    procesadas = 0
    salteadas = 0

    for label_path in label_files:
        stem_label_original = label_path.stem
        stem_label = corregir_stem_duplicado(stem_label_original)
        base, canal_marcado = detectar_canal_en_stem(stem_label)

        if canal_marcado in CANALES_TODOS:
            img_ref_path, stem_ref = buscar_imagen_canal(base, canal_marcado)
        else:
            img_ref_path, stem_ref = buscar_imagen_por_stems([stem_label])

        if img_ref_path is None:
            print(f"[AVISO] No encontré imagen de referencia para etiqueta {label_path.name}")
            salteadas += 1
            continue

        with open(label_path, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]

        if not lines:
            print(f"[AVISO] Etiqueta vacía: {label_path.name}")
            salteadas += 1
            continue

        with Image.open(img_ref_path) as img_ref:
            source_img_size = img_ref.size  # (w, h)

        canales_a_extraer = detectar_canales_disponibles(base, canal_marcado)

        if canal_marcado == "VIRTUAL":
            canales_virtual = [c for c in canales_a_extraer if c == "VIRTUAL"]
            canales_a_extraer = [c for c in CANALES_REALES if canal_tiene_datos(base, c)] + canales_virtual

        safe_base = base.replace("/", "_")
        out_folder = OUTPUT_DIR / f"{safe_base}__marcado_en_{canal_marcado}"
        out_folder.mkdir(parents=True, exist_ok=True)

        metadata = {
            "formato": VERSION_METADATA,
            "autocontenido": True,

            # Trazabilidad de la selección YOLO. Se guarda el nombre del archivo
            # y también el contenido interpretado; el JSON final no necesita el
            # .txt original.
            "etiqueta_yolo": {
                "archivo_fuente": label_path.name,
                "stem_label_original": stem_label_original,
                "stem_label_corregido": stem_label,
                "objetos": serializar_etiquetas_yolo(lines),
            },

            "base_sin_canal": base,
            "canal_marcado_en_yolo": canal_marcado,
            "imagen_referencia_marcado": img_ref_path.name,
            "stem_referencia": stem_ref,
            "imagen_referencia_yolo_w_h": list(source_img_size),
            "canales_detectados_para_extraer": canales_a_extraer,

            "convencion_recorte_matrices": {
                "eje_0": "x de JPG / ping / tiempo",
                "eje_1": "y de JPG / muestra / profundidad",
                "descripcion": (
                    "x JPG -> eje 0/pings de matriz; "
                    "y JPG -> eje 1/muestras de matriz"
                ),
            },

            "nota": (
                "metadata_recortes.json contiene las etiquetas YOLO utilizadas, "
                "los metadatos JSON fuente completos de cada canal, la geometría "
                "y la información necesaria para interpretar cada recorte. "
                "Las copias separadas de los JSON originales son opcionales."
            ),
            "canales": {},
        }

        for canal in canales_a_extraer:
            meta = procesar_canal(
                base=base,
                canal=canal,
                lines=lines,
                out_folder=out_folder,
                canal_marcado=canal_marcado,
                source_img_size=source_img_size,
            )
            if meta is not None:
                metadata["canales"][canal] = meta

        json_out = out_folder / "metadata_recortes.json"
        with open(json_out, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=4, ensure_ascii=False)

        print(f"[OK] {base}: marcado en {canal_marcado}, extraído en {list(metadata['canales'].keys())}")
        procesadas += 1

    print("=" * 80)
    print("RESUMEN")
    print(f"Etiquetas procesadas: {procesadas}")
    print(f"Etiquetas salteadas:  {salteadas}")
    print(f"Salida: {OUTPUT_DIR.resolve()}")
    print("=" * 80)



In [12]:
# Procesar todos los archivos encontrados
main()

BASE_DIR : /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/todas_imagenes_yolo_v4_fish_v6_MAS
LABEL_DIR: /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/todas_imagenes_yolo_v4_fish_v6_MAS/etiquetas_p_cluster_mas_datos/obj_train_data
OUTPUT   : /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/recortes_desde_yolo_segunda_tanda_v2

Etiquetas YOLO encontradas: 80
Buscando etiquetas en: /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/todas_imagenes_yolo_v4_fish_v6_MAS/etiquetas_p_cluster_mas_datos/obj_train_data
Salida: /mnt/Datos/seaman_datos/LUCIANA/Datos_nuevos_jun2026/recortes_desde_yolo_segunda_tanda_v2
Transformación matriz: x JPG -> eje 0/pings, y JPG -> eje 1/muestras
[OK] 08_05_Seaman_Fish_ID_Training_Data_1_P_1: marcado en CH1, extraído en ['CH1', 'CH2']
[AVISO] Etiqueta vacía: 08_05_Seaman_Fish_ID_Training_Data_1_P_1_CH2.txt
[OK] 08_05_Seaman_Fish_ID_Training_Data_1_P_2: marcado en CH1, extraído en ['CH1', 'CH2']
[AVISO] Etiqueta vacía: 08_05_Seaman_Fish_ID_Tra